# Transcode BigWig Signal Analysis

Extracts stranded BigWig Ribo-seq signal for translon DB features and scores:

- body coverage
- 3-nt periodicity
- codon-level signal uniformity
- start-region rise over upstream flank
- post-stop drop-off

The heavy extraction is implemented in `transcode_bigwig_signal_scores.py` so this notebook can run a bounded pilot first and then resume/scale from checkpointed per-sample outputs.


In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib
try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is None:
        matplotlib.use('Agg')
    else:
        _ip.run_line_magic('matplotlib', 'inline')
except Exception:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SCRIPT_DIR = Path.cwd()
if (SCRIPT_DIR / 'transcode_bigwig_signal_scores.py').exists():
    sys.path.insert(0, str(SCRIPT_DIR))
else:
    sys.path.insert(0, str(Path('/Users/jackt/projects/all-RiboSeq/ensembl-genes-nf/pipelines/translon-consensus/scripts')))

from transcode_bigwig_signal_scores import (
    HAS_PYBIGWIG,
    ScoreConfig,
    discover_bigwigs,
    expand_scores_by_tool,
    periodicity_score,
    score_database,
    uniformity_score,
)

# HPC defaults. Override these in-place for local testing.
DB = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/pilot/full_pilot_results/translon_db_rebuild/translon_db/translons.sqlite')
BIGWIG_ROOT = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/pilot/bigwigs')
OUT_DIR = Path('figures_bigwig')
OUT_DIR.mkdir(exist_ok=True)

TOOL_ORDER = ['PRICE', 'RiboTIE', 'ORFQuant', 'iRibo', 'RibORF2']
CLS_ORDER = ['cds', 'non_cds']
CLS_LABELS = {'cds': 'CDS', 'non_cds': 'Non-CDS'}
TOOL_COLORS = {
    'PRICE': '#4C72B0',
    'RiboTIE': '#DD8452',
    'ORFQuant': '#55A868',
    'iRibo': '#C44E52',
    'RibORF2': '#8172B2',
}

print(f'pyBigWig available: {HAS_PYBIGWIG}')
print(f'DB path: {DB}')
print(f'DB exists: {DB.exists()}')
print(f'BigWig root: {BIGWIG_ROOT}')
print(f'BigWig root exists: {BIGWIG_ROOT.exists()}')
print(f'Output dir: {OUT_DIR.resolve()}')


## Quick checks

The scoring functions are deterministic and cheap to test before touching BigWigs.


In [ ]:
test_arr = np.array([10, 0, 0, 8, 0, 0, 6, 0, 0, 4, 0, 0], dtype=float)
flat_arr = np.ones(12, dtype=float)
assert abs(periodicity_score(test_arr) - 1.0) < 1e-6
assert abs(periodicity_score(flat_arr) - 1/3) < 1e-6
assert 0 < uniformity_score(test_arr) <= 1
assert abs(uniformity_score(flat_arr) - 1.0) < 1e-6
print('Metric self-tests passed.')


## BigWig discovery

The discovery code correctly strips full suffixes such as `.forward.bw` and `.reverse.bw`, then keeps only samples with both strands for scoring.


In [ ]:
bw_manifest = discover_bigwigs(BIGWIG_ROOT)
print(f'BigWig samples found: {len(bw_manifest):,}')
if not bw_manifest.empty:
    print(f'Paired stranded BigWigs: {int(bw_manifest.has_pair.sum()):,}')
    display(bw_manifest.head(20))
else:
    print('No BigWigs found yet.')


## Score translon features

Default mode is a bounded pilot: up to 2,000 unique feature/sample signals per sample, checkpointed under `figures_bigwig/signal_score_batches`. This gives usable results across all metrics without loading the full DB or all signals into memory.

When the pilot looks sane, set `max_features_per_sample=None` or run the CLI command printed below with `--full`.


In [ ]:
cfg = ScoreConfig(
    flank_nt=30,
    body_edge_nt=15,
    min_body_nt=30,
    min_total_signal=10.0,
    max_features_per_sample=2000,   # set None for full production scoring
    max_total_features=None,        # optional hard global cap for smoke tests
    overwrite=False,                # existing per-sample checkpoints are reused
)

existing_scores = OUT_DIR / 'translon_signal_scores.tsv.gz'

if HAS_PYBIGWIG and DB.exists() and BIGWIG_ROOT.exists():
    scored_unique, run_summary = score_database(DB, BIGWIG_ROOT, OUT_DIR, cfg)
else:
    print('Skipping extraction because pyBigWig, DB, or BigWig root is unavailable.')
    if existing_scores.exists():
        print(f'Loading existing scores: {existing_scores}')
        scored_unique = pd.read_csv(existing_scores, sep='	')
        run_summary = {'result_rows': len(scored_unique), 'loaded_existing': True}
    else:
        scored_unique = pd.DataFrame()
        run_summary = {'result_rows': 0, 'loaded_existing': False}

scored = expand_scores_by_tool(scored_unique)
print(f'Unique feature/sample score rows: {len(scored_unique):,}')
print(f'Tool-expanded rows for plots: {len(scored):,}')
if not scored_unique.empty:
    display(scored_unique.head())

print('\nFull-scale CLI, resumable from existing sample checkpoints:')
print(
    'python3 transcode_bigwig_signal_scores.py '
    f'--db {DB} --bigwig-root {BIGWIG_ROOT} --out-dir {OUT_DIR} --full'
)



## QC summaries

`coverage_pass` is based on body signal only. Start and stop flank metrics are marked as genomic-flank based because the translon DB currently stores ORF blocks, not full transcript exon models for splice-aware UTR flanks.


In [ ]:
if scored_unique.empty:
    print('No scored features available.')
else:
    summary_cols = [
        'mean_cov', 'body_mean_cov', 'periodicity', 'uniformity',
        'start_rise_ratio', 'stop_dropoff_ratio', 'body_total_signal',
        'upstream_flank_nt', 'downstream_flank_nt'
    ]
    present = [c for c in summary_cols if c in scored_unique]
    display(scored_unique[present].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
    print('\nCoverage pass counts:')
    display(scored_unique['coverage_pass'].value_counts(dropna=False).rename('features'))



## Signal metric distributions


In [ ]:
def tools_present(df):
    return [t for t in TOOL_ORDER if 'source_tool' in df and t in set(df['source_tool'].dropna())]


def plot_metric_by_tool(df, metric, title, ylabel, log=False, finite_positive=False, ylim=None):
    if df.empty or metric not in df:
        print(f'Skipping {metric}: no data.')
        return
    tools = tools_present(df)
    if not tools:
        print(f'Skipping {metric}: no source_tool data.')
        return

    fig, axes = plt.subplots(1, len(CLS_ORDER), figsize=(13, 5), squeeze=False)
    for ax, cls in zip(axes[0], CLS_ORDER):
        sub = df[df['cls'] == cls] if 'cls' in df else df
        positions = list(range(1, len(tools) + 1))
        for pos, tool in zip(positions, tools):
            vals = pd.to_numeric(sub[sub['source_tool'] == tool][metric], errors='coerce').to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            if finite_positive:
                vals = vals[vals > 0]
            if len(vals) < 5:
                continue
            if log:
                vals = np.log10(vals)
            parts = ax.violinplot([vals], positions=[pos], widths=0.7, showmedians=True, showextrema=False)
            for pc in parts['bodies']:
                pc.set_facecolor(TOOL_COLORS.get(tool, '#888'))
                pc.set_alpha(0.7)
            parts['cmedians'].set_color('black')
            parts['cmedians'].set_linewidth(2)
        ax.set_xticks(positions)
        ax.set_xticklabels(tools, rotation=30, ha='right')
        ax.set_title(f'{CLS_LABELS.get(cls, cls)}')
        ax.set_ylabel(ylabel)
        if ylim:
            ax.set_ylim(*ylim)
        ax.yaxis.grid(True, alpha=0.35)
        ax.set_axisbelow(True)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    out = OUT_DIR / f'{metric}_distribution.png'
    fig.savefig(out, dpi=180)
    plt.show()
    print(f'Saved {out}')

if scored.empty:
    print('No scored ORFs available for plots.')
else:
    plot_df = scored[scored.get('coverage_pass', True).astype(bool)].copy() if 'coverage_pass' in scored else scored.copy()
    plot_metric_by_tool(plot_df, 'mean_cov', 'Per-feature mean Ribo-seq coverage', 'log10(mean coverage)', log=True, finite_positive=True)
    plot_metric_by_tool(plot_df, 'periodicity', '3-nt periodicity over ORF body', 'Frame-0 signal fraction', ylim=(0, 1.05))
    plot_metric_by_tool(plot_df, 'uniformity', 'Codon-binned body signal uniformity', '1 / (1 + CV)', ylim=(0, 1.05))
    plot_metric_by_tool(plot_df, 'start_rise_ratio', 'Start-region rise over upstream flank', 'log10(start / upstream)', log=True, finite_positive=True)
    plot_metric_by_tool(plot_df, 'stop_dropoff_ratio', 'Post-stop signal relative to pre-stop signal', 'log10(post-stop / pre-stop)', log=True, finite_positive=True)


## Shared-feature cross-tool comparison


In [ ]:
if scored.empty or 'source_tool' not in scored:
    print('No scored ORFs available for cross-tool comparison.')
else:
    shared_fks = (
        scored.groupby(['feature_key', 'sample_id'])['source_tool']
              .nunique()
              .pipe(lambda s: s[s >= 2])
              .index
    )
    scored_shared = scored[
        scored.set_index(['feature_key', 'sample_id']).index.isin(shared_fks)
    ].copy()
    print(f'Shared feature/sample rows: {len(scored_shared):,}')

    metric = 'periodicity'
    tool_pairs = [(TOOL_ORDER[i], TOOL_ORDER[j]) for i in range(len(TOOL_ORDER)) for j in range(i + 1, len(TOOL_ORDER))]
    valid_pairs = [(a, b) for a, b in tool_pairs if a in set(scored_shared['source_tool']) and b in set(scored_shared['source_tool'])]

    if len(scored_shared) < 10 or not valid_pairs:
        print('Too few shared ORFs for cross-tool scatter.')
    else:
        ncols = min(3, len(valid_pairs))
        nrows = (len(valid_pairs) + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
        for idx, (t1, t2) in enumerate(valid_pairs):
            ax = axes[idx // ncols][idx % ncols]
            sub1 = scored_shared[scored_shared['source_tool'] == t1][['feature_key', 'sample_id', metric]].rename(columns={metric: 'm1'})
            sub2 = scored_shared[scored_shared['source_tool'] == t2][['feature_key', 'sample_id', metric]].rename(columns={metric: 'm2'})
            merged = sub1.merge(sub2, on=['feature_key', 'sample_id']).dropna()
            if len(merged) < 3:
                ax.set_visible(False)
                continue
            ax.scatter(merged['m1'], merged['m2'], alpha=0.35, s=10, color='#555')
            ax.plot([0, 1], [0, 1], 'r--', lw=1)
            ax.axhline(1/3, color='grey', lw=0.7, linestyle=':')
            ax.axvline(1/3, color='grey', lw=0.7, linestyle=':')
            corr = merged['m1'].corr(merged['m2'])
            ax.set_xlabel(f'{t1} {metric}')
            ax.set_ylabel(f'{t2} {metric}')
            ax.set_title(f'{t1} vs {t2}\n(r={corr:.2f}, n={len(merged):,})')
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
        for idx in range(len(valid_pairs), nrows * ncols):
            axes[idx // ncols][idx % ncols].set_visible(False)
        fig.suptitle('Cross-tool periodicity comparison for shared feature_keys', fontsize=12)
        plt.tight_layout()
        out = OUT_DIR / 'cross_tool_periodicity_scatter.png'
        fig.savefig(out, dpi=180)
        plt.show()
        print(f'Saved {out}')



## CDS recall versus signal


In [ ]:
if scored.empty or not DB.exists():
    print('No scored ORFs or DB unavailable; skipping CDS recall plot.')
else:
    con = sqlite3.connect(DB)
    ref = pd.read_sql_query('SELECT feature_key, stop_excluded_feature_key FROM reference_cds', con)
    con.close()
    ref_fks = set(ref['feature_key'].dropna()) | set(ref['stop_excluded_feature_key'].dropna())

    scored_cds = scored[scored['cls'] == 'cds'].copy()
    scored_cds['recalled'] = scored_cds['feature_key'].isin(ref_fks)
    recalled_cov = scored_cds[scored_cds['recalled'] & (scored_cds['mean_cov'] > 0)]['mean_cov'].dropna()
    not_recalled_cov = scored_cds[~scored_cds['recalled'] & (scored_cds['mean_cov'] > 0)]['mean_cov'].dropna()
    print(f'Scored CDS rows: recalled={len(recalled_cov):,}, not-recalled={len(not_recalled_cov):,}')

    if len(recalled_cov) and len(not_recalled_cov):
        fig, ax = plt.subplots(figsize=(8, 5))
        bins = np.logspace(-2, 4, 60)
        ax.hist(recalled_cov, bins=bins, color='#2196F3', alpha=0.65, label=f'Recalled (n={len(recalled_cov):,})', density=True)
        ax.hist(not_recalled_cov, bins=bins, color='#E53935', alpha=0.65, label=f'Not recalled (n={len(not_recalled_cov):,})', density=True)
        ax.set_xscale('log')
        ax.set_xlabel('Mean coverage (reads per nt)')
        ax.set_ylabel('Density')
        ax.set_title('CDS mean coverage: reference CDS recalled vs not recalled')
        ax.legend()
        ax.yaxis.grid(True, alpha=0.35)
        ax.set_axisbelow(True)
        plt.tight_layout()
        out = OUT_DIR / 'cds_recall_vs_coverage.png'
        fig.savefig(out, dpi=180)
        plt.show()
        print(f'Saved {out}')
        print(f'Median coverage: recalled={np.median(recalled_cov):.3f}; not-recalled={np.median(not_recalled_cov):.3f}')
